# Lesson 2 - Object Oriented Programming - Exercices

Les trois exercices du cours, implementes dans le style du support :

1. **Comptes bancaires** - heritage, methode abstraite redefinie, composition, historique
2. **Formes geometriques** - polymorphisme, `__repr__`, 100 objets aleatoires
3. **Fractional** - surcharge des operateurs arithmetiques et de comparaison

---
## Rappel express du cours

| Concept | En une phrase |
| :-- | :-- |
| **Classe** | un modele decrivant des attributs (donnees) et des methodes (comportements) |
| **Objet** | une instance concrete creee a partir de la classe |
| **`__init__`** | le constructeur, appele automatiquement a chaque instanciation |
| **`self`** | reference vers l'objet courant, toujours le 1er parametre d'une methode |
| **Encapsulation** | regrouper donnees et code, et proteger l'acces (`_protege`, `__prive`) |
| **Heritage** | une classe fille reprend les attributs et methodes de la classe mere |
| **Polymorphisme** | le meme nom de methode, un comportement different selon la classe |
| **Composition** | un attribut peut etre un objet d'une autre classe |

In [ ]:
class Person:
    """Le squelette minimal d'une classe, comme dans le support de cours."""

    def __init__(self, name, age):   # constructeur, 2 parametres
        self.name = name             # attribut
        self.age = age               # attribut

    def greet(self):                 # methode
        print(f"Hello, my name is {self.name} and I am {self.age} years old.")


if __name__ == "__main__":
    person1 = Person("Alice", 30)
    person2 = Person("Bob", 25)
    person1.greet()
    person2.greet()

### Public, protege, prive

Tout est **public par defaut** en Python. Les underscores sont une convention,
pas une barriere :

- `_protected` : "ne touche pas depuis l'exterieur", mais rien ne l'empeche
- `__private` : Python renomme l'attribut en `_MaClasse__private` (*name mangling*),
  ce qui le rend difficile a atteindre par accident, mais pas impossible

In [ ]:
class Test:
    varPublic = 10
    _varProtected = 20
    __varPrivate = 30


t = Test()
print(t.varPublic)        # 10
print(t._varProtected)    # 20  -> accessible malgre la convention

try:
    print(t.__varPrivate)
except AttributeError as error:
    print("AttributeError :", error)

# L'attribut existe toujours, sous un nom transforme :
print(t._Test__varPrivate)   # 30

---
# Exercice 1 - Comptes bancaires

Le diagramme UML du cours :

```
                    BankAccount            <- classe abstraite (en italique)
                    owner : String
                    balance : Dollars
                    deposit(amount)
                    withdrawal(amount)     <- methode abstraite (en italique)
                          ^
             +------------+------------+
             |                         |
    CheckingAccount              SavingsAccount
    insufficientFundsFee         annualInterestRate
    processCheck(check)          depositMonthlyInterest()
    withdrawal(amount)           withdrawal(amount)
```

Deux elements sont **en italique** dans le diagramme : la classe `BankAccount`
et sa methode `withdrawal`. En UML, l'italique signale l'**abstrait** : on
n'instancie jamais un `BankAccount` directement, et chaque type de compte
**doit** definir sa propre facon de retirer de l'argent.

A cela s'ajoute la consigne : **un historique des transactions**.

### La classe `Transaction`

L'historique est une illustration directe de la **composition** : un attribut
de `BankAccount` est une liste d'objets d'une autre classe.

In [1]:
from datetime import datetime


class Transaction:
    """Une operation enregistree dans l'historique d'un compte."""

    def __init__(self, kind, amount, balance_after):
        self.kind = kind                    # "deposit", "withdrawal", "fee", ...
        self.amount = amount
        self.balance_after = balance_after
        self.date = datetime.now()

    def __repr__(self):
        return (f"{self.date:%H:%M:%S} | {self.kind:<10} "
                f"{self.amount:>9.2f} | balance = {self.balance_after:>9.2f}")


class Check:
    """Un cheque a encaisser. Composition, la aussi : processCheck en recoit un."""

    def __init__(self, amount, payee):
        self.amount = amount
        self.payee = payee

    def __repr__(self):
        return f"Check({self.amount:.2f} to {self.payee})"

### La classe mere `BankAccount`

Le solde est **protege** (`_balance`) : on ne le modifie que par `deposit()` et
`withdrawal()`, ce qui garantit qu'aucune operation n'echappe a l'historique.
C'est exactement l'idee d'encapsulation, et on lit la valeur avec un accesseur
`get_balance()`, comme le `get_count()` du compteur vu en cours.

`withdrawal()` leve `NotImplementedError` : c'est la facon la plus simple de
rendre une methode abstraite. Le support ecrit `pass` dans la methode du parent,
mais `pass` renvoie silencieusement `None` et laisse passer l'oubli, alors que
`NotImplementedError` le signale immediatement.

In [2]:
class BankAccount:
    """Classe de base abstraite : withdrawal() doit etre redefinie."""

    def __init__(self, owner, balance=0.0):
        self.owner = owner
        self._balance = balance          # protege : voir _record()
        self._transactions = []          # composition : une liste de Transaction
        if balance:
            self._record("open", balance)

    # ---------------------------------------------------------- lecture
    def get_balance(self):
        return self._balance

    def history(self):
        return list(self._transactions)  # une copie : l'historique reste intact

    # ---------------------------------------------------------- ecriture
    def _record(self, kind, amount):
        """Ajoute une ligne a l'historique. Protege : usage interne."""
        self._transactions.append(Transaction(kind, amount, self._balance))

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("The deposited amount must be positive")
        self._balance += amount
        self._record("deposit", amount)
        return self._balance

    def withdrawal(self, amount):
        """Methode abstraite : chaque type de compte a ses propres regles."""
        raise NotImplementedError("Each account type must define withdrawal()")

    def __repr__(self):
        return f"{type(self).__name__}({self.owner!r}, balance={self._balance:.2f})"

### Les deux classes filles

`super().__init__(owner, balance)` appelle le constructeur du parent : sans lui,
`owner`, `_balance` et `_transactions` ne seraient jamais crees.

Les deux classes redefinissent `withdrawal()` avec des regles differentes :
le compte courant autorise le decouvert mais prend des frais, le compte epargne
le refuse. C'est du **method overriding**, et c'est ce qui rend le polymorphisme
possible.

In [4]:
class CheckingAccount(BankAccount):
    """Compte courant : le decouvert est accepte, mais facture."""

    def __init__(self, owner, balance=0.0, insufficientFundsFee=30.0):
        super().__init__(owner, balance)          # indispensable
        self.insufficientFundsFee = insufficientFundsFee

    def withdrawal(self, amount):
        if amount <= 0:
            raise ValueError("The withdrawn amount must be positive")

        if amount > self._balance:
            self._balance -= self.insufficientFundsFee
            self._record("fee", self.insufficientFundsFee)
            return False                          # retrait refuse

        self._balance -= amount
        self._record("withdrawal", amount)
        return True

    def processCheck(self, checkToProcess):
        """Encaisse un cheque : un retrait, avec sa propre ligne d'historique."""
        accepted = self.withdrawal(checkToProcess.amount)
        if accepted:
            self._transactions[-1].kind = f"check:{checkToProcess.payee}"
        return accepted


class SavingsAccount(BankAccount):
    """Compte epargne : pas de decouvert, mais des interets."""

    def __init__(self, owner, balance=0.0, annualInterestRate=0.02):
        super().__init__(owner, balance)
        self.annualInterestRate = annualInterestRate

    def withdrawal(self, amount):
        if amount <= 0:
            raise ValueError("The withdrawn amount must be positive")
        if amount > self._balance:
            raise ValueError("Insufficient funds on a savings account")

        self._balance -= amount
        self._record("withdrawal", amount)
        return True

    def depositMonthlyInterest(self):
        interest = self._balance * self.annualInterestRate / 12
        self._balance += interest
        self._record("interest", interest)
        return interest

### Exemples d'utilisation

In [5]:
if __name__ == "__main__":
    checking = CheckingAccount("Alice", 500.0)
    savings = SavingsAccount("Bob", 2000.0, annualInterestRate=0.03)

    checking.deposit(250.0)
    print("Retrait de 100 :", checking.withdrawal(100.0))    # True
    print("Retrait de 900 :", checking.withdrawal(900.0))    # False -> frais
    checking.processCheck(Check(120.0, "EDF"))

    savings.depositMonthlyInterest()
    savings.withdrawal(500.0)

    print()
    print(checking)
    for transaction in checking.history():
        print("   ", transaction)

    print()
    print(savings)
    for transaction in savings.history():
        print("   ", transaction)

Retrait de 100 : True
Retrait de 900 : False

CheckingAccount('Alice', balance=500.00)
    09:28:46 | open          500.00 | balance =    500.00
    09:28:46 | deposit       250.00 | balance =    750.00
    09:28:46 | withdrawal    100.00 | balance =    650.00
    09:28:46 | fee            30.00 | balance =    620.00
    09:28:46 | check:EDF     120.00 | balance =    500.00

SavingsAccount('Bob', balance=1505.00)
    09:28:46 | open         2000.00 | balance =   2000.00
    09:28:46 | interest        5.00 | balance =   2005.00
    09:28:46 | withdrawal    500.00 | balance =   1505.00


### Polymorphisme et classe abstraite

Le meme appel `account.withdrawal(...)` declenche deux comportements differents.
C'est tout l'interet : le code qui manipule les comptes n'a pas besoin de savoir
de quel type ils sont.

In [ ]:
accounts = [CheckingAccount("Alice", 100.0), SavingsAccount("Bob", 100.0)]

for account in accounts:
    try:
        account.withdrawal(500.0)                 # meme appel
        print(f"{type(account).__name__:<16} -> retrait effectue")
    except ValueError as error:
        print(f"{type(account).__name__:<16} -> refuse : {error}")
    print(f"{'':<16}    solde = {account.get_balance():.2f}")

# Et la classe mere reste inutilisable telle quelle :
try:
    BankAccount("Charlie", 50.0).withdrawal(10.0)
except NotImplementedError as error:
    print("\nBankAccount    -> NotImplementedError :", error)

> **Pour aller plus loin.** Le module standard `abc` rend l'abstraction stricte :
> en heritant de `ABC` et en decorant la methode avec `@abstractmethod`, Python
> refuse purement et simplement d'instancier la classe mere. Avec
> `NotImplementedError`, l'erreur n'arrive qu'au moment de l'appel.

---
# Exercice 2 - Formes geometriques

Consigne : implementer differentes formes, ajouter une methode `computeArea()`,
puis creer une liste de 100 formes aleatoires, afficher leur representation et
calculer la somme des aires.

C'est le meme schema que l'exercice 1 : une classe mere `Shape` qui definit le
contrat, des classes filles qui l'implementent chacune a leur maniere.

In [ ]:
import math
import random


class Shape:
    """Classe de base : toute forme sait calculer son aire."""

    def computeArea(self):
        raise NotImplementedError("Each shape must define computeArea()")

    def __repr__(self):
        return f"{type(self).__name__}(area={self.computeArea():.2f})"


class Rectangle(Shape):
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def computeArea(self):
        return self.width * self.height

    def __repr__(self):
        return (f"Rectangle({self.width:.1f} x {self.height:.1f}, "
                f"area={self.computeArea():.2f})")


class Square(Rectangle):
    """Un carre EST un rectangle : on herite, on ne recopie pas."""

    def __init__(self, side):
        super().__init__(side, side)

    def __repr__(self):
        return f"Square({self.width:.1f}, area={self.computeArea():.2f})"


class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def computeArea(self):
        return math.pi * self.radius ** 2

    def __repr__(self):
        return f"Circle(r={self.radius:.1f}, area={self.computeArea():.2f})"


class Triangle(Shape):
    def __init__(self, base, height):
        self.base = base
        self.height = height

    def computeArea(self):
        return self.base * self.height / 2

    def __repr__(self):
        return (f"Triangle({self.base:.1f} x {self.height:.1f}, "
                f"area={self.computeArea():.2f})")

### 100 formes aleatoires

`random.choice` tire une **classe** au hasard, puis on l'instancie. Le
dictionnaire associe a chaque classe la facon de tirer ses parametres.

In [ ]:
def random_shape():
    """Cree une forme aleatoire parmi les quatre types disponibles."""
    builders = {
        Rectangle: lambda: Rectangle(random.uniform(1, 10), random.uniform(1, 10)),
        Square: lambda: Square(random.uniform(1, 10)),
        Circle: lambda: Circle(random.uniform(1, 5)),
        Triangle: lambda: Triangle(random.uniform(1, 10), random.uniform(1, 10)),
    }
    return random.choice(list(builders.values()))()


random.seed(0)                      # pour obtenir toujours le meme tirage
shapes = [random_shape() for _ in range(100)]

print("Les 10 premieres formes :")
for shape in shapes[:10]:
    print("   ", shape)

total = sum(shape.computeArea() for shape in shapes)
print(f"\nNombre de formes : {len(shapes)}")
print(f"Somme des aires  : {total:.2f}")

Le `sum(shape.computeArea() for shape in shapes)` est du polymorphisme pur :
une seule ligne, quatre implementations differentes appelees derriere.

### Compter et regrouper par type

In [ ]:
counts = {}
areas = {}
for shape in shapes:
    name = type(shape).__name__
    counts[name] = counts.get(name, 0) + 1
    areas[name] = areas.get(name, 0.0) + shape.computeArea()

print(f"{'Type':<12}{'Nombre':>8}{'Aire totale':>14}{'Aire moyenne':>15}")
print("-" * 49)
for name in sorted(counts):
    print(f"{name:<12}{counts[name]:>8}{areas[name]:>14.2f}"
          f"{areas[name] / counts[name]:>15.2f}")

# La plus grande forme du lot
largest = max(shapes, key=lambda shape: shape.computeArea())
print(f"\nPlus grande forme : {largest}")

---
# Exercice 3 - La classe `Fractional`

Consigne du cours :

```python
frac1 = Fractional(1, 2)
frac2 = Fractional(3, 4)
print(frac1 + frac2)     # 5/4
print(frac1 > frac2)     # False
frac3 = Fractional(42, 0)   # ???
```

Le `???` est la vraie question de l'exercice : **un denominateur nul n'a pas de
sens**. On refuse la construction en levant une exception, plutot que de creer
un objet invalide qui plantera plus tard, loin de sa cause.

### Les methodes magiques a implementer

| Operateur | Methode |
| :-- | :-- |
| `+` `-` `*` `/` | `__add__` `__sub__` `__mul__` `__truediv__` |
| `<` `>` `<=` `>=` | `__lt__` `__gt__` `__le__` `__ge__` |
| `==` `!=` | `__eq__` `__ne__` |
| `print()` | `__repr__` (utilise a defaut de `__str__`) |

Deux points de methode :

- **Reduire des la construction.** `Fractional(2, 4)` doit devenir `1/2`, sinon
  `==` compare des fractions ecrites differemment mais egales.
- **Comparer par produit croise.** `a/b < c/d` equivaut a `a*d < c*b` quand les
  denominateurs sont positifs. On evite la division flottante et ses arrondis.

In [ ]:
from math import gcd


def _as_fractional(value):
    """Accepte un Fractional ou un entier, refuse le reste."""
    if isinstance(value, Fractional):
        return value
    if isinstance(value, int):
        return Fractional(value, 1)
    raise TypeError(f"Unsupported operand type: {type(value).__name__}")


class Fractional:
    """Une fraction rationnelle, toujours stockee sous forme reduite."""

    def __init__(self, numerator, denominator=1):
        if denominator == 0:
            raise ZeroDivisionError("A fraction cannot have a zero denominator")

        # Le signe est porte par le numerateur : 1/-2 devient -1/2
        if denominator < 0:
            numerator, denominator = -numerator, -denominator

        divisor = gcd(abs(numerator), denominator)
        self.numerator = numerator // divisor
        self.denominator = denominator // divisor

    # ------------------------------------------------------- affichage
    def __repr__(self):
        if self.denominator == 1:
            return str(self.numerator)
        return f"{self.numerator}/{self.denominator}"

    # ------------------------------------------------------ arithmetique
    def __add__(self, other):
        other = _as_fractional(other)
        return Fractional(
            self.numerator * other.denominator + other.numerator * self.denominator,
            self.denominator * other.denominator,
        )

    def __sub__(self, other):
        other = _as_fractional(other)
        return Fractional(
            self.numerator * other.denominator - other.numerator * self.denominator,
            self.denominator * other.denominator,
        )

    def __mul__(self, other):
        other = _as_fractional(other)
        return Fractional(self.numerator * other.numerator,
                          self.denominator * other.denominator)

    def __truediv__(self, other):
        other = _as_fractional(other)
        if other.numerator == 0:
            raise ZeroDivisionError("Division by a zero fraction")
        return Fractional(self.numerator * other.denominator,
                          self.denominator * other.numerator)

    # ------------------------------------------------------ comparaison
    def __eq__(self, other):
        other = _as_fractional(other)
        return (self.numerator == other.numerator
                and self.denominator == other.denominator)

    def __ne__(self, other):
        # Python 3 le deduit deja de __eq__, on l'ecrit pour suivre le cours
        return not self == other

    def __lt__(self, other):
        other = _as_fractional(other)
        return self.numerator * other.denominator < other.numerator * self.denominator

    def __gt__(self, other):
        other = _as_fractional(other)
        return self.numerator * other.denominator > other.numerator * self.denominator

    def __le__(self, other):
        return not self > other

    def __ge__(self, other):
        return not self < other

### Les exemples de l'enonce

In [ ]:
frac1 = Fractional(1, 2)
frac2 = Fractional(3, 4)

print("frac1 + frac2 =", frac1 + frac2)     # 5/4
print("frac1 - frac2 =", frac1 - frac2)     # -1/4
print("frac1 * frac2 =", frac1 * frac2)     # 3/8
print("frac1 / frac2 =", frac1 / frac2)     # 2/3
print("frac1 > frac2 =", frac1 > frac2)     # False
print("frac1 < frac2 =", frac1 < frac2)     # True

# Reduction automatique et signe normalise
print("\nFractional(2, 4)   =", Fractional(2, 4))     # 1/2
print("Fractional(6, 3)   =", Fractional(6, 3))       # 2
print("Fractional(1, -2)  =", Fractional(1, -2))      # -1/2
print("Fractional(2, 4) == Fractional(1, 2) :", Fractional(2, 4) == Fractional(1, 2))

# Le cas Fractional(42, 0)
try:
    frac3 = Fractional(42, 0)
except ZeroDivisionError as error:
    print("\nFractional(42, 0) -> ZeroDivisionError :", error)

### Melanger fractions et entiers

Grace au `_as_fractional()`, les operateurs acceptent aussi des entiers.
Attention toutefois : cela ne fonctionne que si la **fraction est a gauche**.
`1 + frac1` demanderait `__radd__`, la version "a droite" de `__add__`.

In [ ]:
print("frac1 + 1 =", frac1 + 1)       # 3/2
print("frac1 * 2 =", frac1 * 2)       # 1
print("frac1 == Fractional(2, 4) :", frac1 == Fractional(2, 4))

try:
    print(1 + frac1)
except TypeError as error:
    print("1 + frac1 -> TypeError :", error)

try:
    print(frac1 / Fractional(0, 5))
except ZeroDivisionError as error:
    print("frac1 / (0/5) -> ZeroDivisionError :", error)

try:
    print(frac1 + "1/2")
except TypeError as error:
    print('frac1 + "1/2" -> TypeError :', error)

---
## Recapitulatif

Un test global qui rejoue les trois exercices.

In [ ]:
# 1. Comptes bancaires
account = CheckingAccount("Alice", 500.0)
account.withdrawal(100.0)
print("1. solde apres retrait   ->", account.get_balance())
print("1. lignes d'historique   ->", len(account.history()))

# 2. Formes
random.seed(0)
shapes = [random_shape() for _ in range(100)]
print("2. nombre de formes      ->", len(shapes))
print("2. somme des aires       ->", round(sum(s.computeArea() for s in shapes), 2))

# 3. Fractional
print("3. 1/2 + 3/4             ->", Fractional(1, 2) + Fractional(3, 4))
print("3. 1/2 > 3/4             ->", Fractional(1, 2) > Fractional(3, 4))
print("3. Fractional(2, 4)      ->", Fractional(2, 4))